# Parsing Allen Institute Taxonomy (AIT) `.h5ad` files

This notebook walks through parsing AIT taxonomy files (BICAN / HMBA basal-ganglia,
produced with the [`scrattch`](https://alleninstitute.github.io/scrattch/) toolkit)
step by step. It mirrors `bkbit/data_translators/ait_taxonomy_parser.py`.

**Key idea:** an `.h5ad` file is an HDF5 container. The taxonomy lives in the tiny
`uns` group; the expression matrix `X` is what makes the files huge (30–105 GB).
We read **only** the taxonomy groups — over HTTP range requests — so nothing is
downloaded in full and `X` is never loaded.

## 1. Install dependencies
`anndata` + `h5py` to read HDF5, `fsspec` + `aiohttp` for lazy remote reads.

In [1]:
%pip install anndata h5py fsspec aiohttp pandas


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and file URLs

In [45]:
import fsspec
import h5py
import pandas as pd
from anndata.io import read_elem

BASE = (
    "https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com"
    "/HMBA/BasalGanglia/BICAN_05072025_pre-print_release"
)
URLS = {
    "Human": f"{BASE}/Human_HMBA_basalganglia_AIT_pre-print.h5ad",
    "Macaque": f"{BASE}/Macaque_HMBA_basalganglia_AIT_pre-print.h5ad",
    "Marmoset": f"{BASE}/Marmoset_HMBA_basalganglia_AIT_pre-print.h5ad",
}

# Pick one to explore. Marmoset is the smallest (~30 GB); we still never download it.
SPECIES = "Human"
url = URLS[SPECIES]
url

'https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com/HMBA/BasalGanglia/BICAN_05072025_pre-print_release/Human_HMBA_basalganglia_AIT_pre-print.h5ad'

## 3. Open the remote file lazily (no download)
`fsspec` opens the URL as a file-like object backed by HTTP range requests. `h5py`
reads only the HDF5 index; group data is fetched on demand when accessed.

In [46]:
fileobj = fsspec.open(url, block_size=8 * 1024 * 1024).open()
h5 = h5py.File(fileobj, "r")

# Top-level groups. X / layers / raw are the big expression matrices — we ignore them.
list(h5.keys())

['X', 'layers', 'obs', 'obsm', 'obsp', 'raw', 'uns', 'var', 'varm', 'varp']

## 4. Explore the `uns` group (where the taxonomy lives)

In [47]:
for k in h5["uns"].keys():
    obj = h5["uns"][k]
    if isinstance(obj, h5py.Group):
        print(f"{k}/  (group) -> {list(obj.keys())[:8]}")
    else:
        print(f"{k}: shape={obj.shape} dtype={obj.dtype}")

AIT117_MapMyCells_Flat_Subclass_label_colors: shape=(28,) dtype=object
AIT117_MapMyCells_Group_label_colors: shape=(48,) dtype=object
AIT117_MapMyCells_Subclass_label_colors: shape=(28,) dtype=object
AIT193_MapMyCells_Flat_Subclass_label_colors: shape=(33,) dtype=object
AIT193_MapMyCells_Neighborhood_label_colors: shape=(4,) dtype=object
AIT193_MapMyCells_Subclass_label_colors: shape=(33,) dtype=object
batch_condition: shape=(1,) dtype=object
cluster_algorithm: shape=() dtype=object
cluster_info/  (group) -> ['CL:ID_class', 'CL:ID_group', 'CL:ID_neighborhood', 'CL:ID_subclass', 'Class', 'Cluster', 'Group', 'Neighborhood']
dataset_purl: shape=() dtype=object
default_embedding: shape=() dtype=object
filter/  (group) -> ['standard']
gene_annotation_version: shape=() dtype=object
hierarchy/  (group) -> ['Class', 'Group', 'Neighborhood', 'Subclass', 'cluster_id']
hvg/  (group) -> ['flavor']
leiden_scVI/  (group) -> ['params']
log1p/  (group) -> []
mode: shape=() dtype=object
neighbors/  (gr

### 4a. Taxonomy levels from `uns/hierarchy`
A dict of `level -> position`; sorting by position gives root → leaf order.

In [48]:
hierarchy = read_elem(h5["uns"]["hierarchy"])
levels = [name for name, _ in sorted(hierarchy.items(), key=lambda kv: int(kv[1]))]
print("hierarchy:", hierarchy)
print("levels (root -> leaf):", levels)

hierarchy: {'Class': 1, 'Group': 3, 'Neighborhood': 0, 'Subclass': 2, 'cluster_id': 4}
levels (root -> leaf): ['Neighborhood', 'Class', 'Subclass', 'Group', 'cluster_id']


### 4b. The taxonomy table from `uns/cluster_info`
One row per leaf cluster, with the full ancestor path plus per-level accessions,
CL ontology IDs, and colors. `read_elem` decodes categoricals back to strings and
sets the file's `_index` (`cell_label`) as the DataFrame index.

In [49]:
cluster_info = read_elem(h5["uns"]["cluster_info"])
print("shape:", cluster_info.shape, "| index name:", cluster_info.index.name)
cluster_info[["Neighborhood", "Class", "Subclass", "Group", "Cluster",
              "cluster_id", "accession_group", "CL:ID_group", "color_hex_group"]].head()

shape: (453, 103) | index name: None


,Neighborhood,Class,Subclass,Group,Cluster,cluster_id,accession_group,CL:ID_group,color_hex_group
AAACAGCCAAATGCCC-2362_A05,Nonneuron,Vascular,VLMC,VLMC,Human-451,Human-451,CS20250428_GROUP_0062,CL:4023051,#c3ec86
AAACAGCCAATTGAGA-2362_A05,Nonneuron,OPC-Oligo,Oligodendrocyte,Oligo OPALIN,Human-1,Human-1,CS20250428_GROUP_0025,CL:0000128,#488edc
AAACAGCCAGCATGTC-2362_A05,Nonneuron,Immune,Microglia,Microglia,Human-153,Human-153,CS20250428_GROUP_0019,CL:0000129,#cbfc1e
AAACAGCCATTGTGGC-2362_A05,Nonneuron,Astro-Epen,Astrocyte,Astrocyte,Human-14,Human-14,CS20250428_GROUP_0039,CL:0000127,#e16c95
AAACATGCAGTAGGAT-2362_A05,Nonneuron,Astro-Epen,Astrocyte,Astrocyte,Human-159,Human-159,CS20250428_GROUP_0039,CL:0000127,#e16c95


> **Note:** `cell_label` (the index) is a *representative cell barcode* per cluster,
> not the cluster ID. Use `cluster_id` / `Cluster` to identify a cluster.

In [50]:
cluster_info["Cluster"].nunique()

453

### 4b-i. Checking what `_index` is set to
`_index` is an HDF5 attribute on a dataframe group that records which stored
column becomes the pandas index on read. You can inspect it directly on the raw
group, or read it off the loaded DataFrame's `index.name`. This works on any
dataframe group (`obs`, `var`, `uns/cluster_info`, ...).

In [51]:
grp = h5["uns"]["cluster_info"]  # any dataframe group: h5["obs"], h5["var"], ...

# Method 1: read the raw HDF5 attribute (often bytes -> decode).
idx = grp.attrs["_index"]
idx = idx.decode() if isinstance(idx, bytes) else idx
print("_index attribute:", idx)

# See it alongside the data columns and encoding type:
print("all attrs:", dict(grp.attrs))

# Method 2: on the already-loaded DataFrame, it's the index name.
print("cluster_info.index.name:", cluster_info.index.name)

_index attribute: _index
all attrs: {'_index': '_index', 'column-order': array(['Neighborhood', 'Class', 'Subclass', 'Group', 'Cluster',
       'cluster_id', 'cell_type_ontology_term', 'load_id', 'donor_id',
       'assay', 'assay_ontology_term_id', 'organism',
       'organism_ontology_term_id', 'development_stage',
       'anatomical_region', 'anatomical_region_merged',
       'anatomical_region_ontology_term_id',
       'brain_region_ontology_term_id', 'self_reported_sex',
       'self_reported_sex_ontology_term_id', 'self_reported_ethnicity',
       'self_reported_ethnicity_ontology_term_id', 'disease',
       'disease_ontology_term_id', 'suspension_type', 'is_primary_data',
       'atac_confidently_mapped_read_pairs',
       'atac_fraction_of_genome_in_peaks',
       'atac_fraction_of_high_quality_fragments_in_cells',
       'atac_fraction_of_high_quality_fragments_overlapping_tss',
       'atac_fraction_of_high_quality_fragments_overlapping_peaks',
       'atac_fraction_of_transp

### 4c. Node counts per level

In [52]:
for lvl in levels:
    col = "Cluster" if lvl == "cluster_id" else lvl
    print(f"{lvl:<14} {cluster_info[col].nunique()} nodes")

Neighborhood   4 nodes
Class          12 nodes
Subclass       36 nodes
Group          60 nodes
cluster_id     453 nodes


## 5. Same thing via the `AITTaxonomy` helper
The steps above are packaged in `bkbit/data_translators/ait_taxonomy_parser.py`.
Run this from the repo root so the import resolves.

In [53]:
from bkbit.data_translators.ait_taxonomy_parser import AITTaxonomy

# load_obs=False skips the large per-cell table — recommended for remote files.
tax = AITTaxonomy.from_file(url, load_obs=False)
print(tax.summary())

title:           Human_HMBA_basalganglia_consensus_AIT
schema_version:  v1.0
reference_genome:GRCh38
levels:          Neighborhood > Class > Subclass > Group > cluster_id
leaf clusters:   453
  Neighborhood   4 nodes
  Class          12 nodes
  Subclass       36 nodes
  Group          60 nodes
  cluster_id     453 nodes
genes (var):     36,601


In [54]:
# Parent -> child tree edges derived from each cluster's ancestor path.
edges = tax.edges()
print(f"{len(edges)} edges; first 6:")
for e in edges[:6]:
    print("  ", e[0], e[1], "->", e[2], e[3])

561 edges; first 6:
   Neighborhood Nonneuron -> Class Vascular
   Class Vascular -> Subclass VLMC
   Subclass VLMC -> Group VLMC
   Group VLMC -> cluster_id Human-451
   Neighborhood Nonneuron -> Class OPC-Oligo
   Class OPC-Oligo -> Subclass Oligodendrocyte


In [55]:
import os

OUT_DIR = "bkbit/data/ait_output"  # relative to the repo root
os.makedirs(OUT_DIR, exist_ok=True)
csv_path = f"{OUT_DIR}/{SPECIES}_taxonomy.csv"
pkl_path = f"{OUT_DIR}/{SPECIES}_cluster_info.pkl"

tax.to_csv(csv_path)
tax.cluster_info.to_pickle(pkl_path)
print("wrote:", csv_path, "and", pkl_path)

wrote: bkbit/data/ait_output/Human_taxonomy.csv and bkbit/data/ait_output/Human_cluster_info.pkl


In [56]:
import os

os.makedirs("ait_output", exist_ok=True)
csv_path = f"ait_output/{SPECIES}_taxonomy.csv"
pkl_path = f"ait_output/{SPECIES}_cluster_info.pkl"

tax.to_csv(csv_path)
tax.cluster_info.to_pickle(pkl_path)
print("wrote:", csv_path, "and", pkl_path)

wrote: ait_output/Human_taxonomy.csv and ait_output/Human_cluster_info.pkl


In [57]:
os.makedirs("bkbit/data/ait_output", exist_ok=True)
for sp, u in URLS.items():
    t = AITTaxonomy.from_file(u, load_obs=False)
    t.to_csv(f"bkbit/data/ait_output/{sp}_taxonomy.csv")
    t.cluster_info.to_pickle(f"bkbit/data/ait_output/{sp}_cluster_info.pkl")
    print(f"{sp}: {t.cluster_info.shape[0]} clusters -> saved")

Human: 453 clusters -> saved
Macaque: 388 clusters -> saved
Marmoset: 594 clusters -> saved


In [59]:
for k in h5["obs"].keys():
    obj = h5["obs"][k]
    if isinstance(obj, h5py.Group):
        print(f"{k}/  (group) -> {list(obj.keys())[:8]}")
    else:
        print(f"{k}: shape={obj.shape} dtype={obj.dtype}")

CL:ID_class/  (group) -> ['categories', 'codes']
CL:ID_group/  (group) -> ['categories', 'codes']
CL:ID_neighborhood/  (group) -> ['categories', 'codes']
CL:ID_subclass/  (group) -> ['categories', 'codes']
Class/  (group) -> ['categories', 'codes']
Cluster/  (group) -> ['categories', 'codes']
Group/  (group) -> ['categories', 'codes']
Neighborhood/  (group) -> ['categories', 'codes']
Subclass/  (group) -> ['categories', 'codes']
_index: shape=(1034819,) dtype=object
accession_class/  (group) -> ['categories', 'codes']
accession_group/  (group) -> ['categories', 'codes']
accession_neighborhood/  (group) -> ['categories', 'codes']
accession_subclass/  (group) -> ['categories', 'codes']
alignment_job_database/  (group) -> ['categories', 'codes']
anatomical_region/  (group) -> ['categories', 'codes']
anatomical_region_merged/  (group) -> ['categories', 'codes']
anatomical_region_ontology_term_id/  (group) -> ['categories', 'codes']
assay/  (group) -> ['categories', 'codes']
assay_ontology_

In [60]:
# Extract donor_ids from obs. donor_id is a categorical, so:
#   - unique donors      = the `categories` array (cheap; no per-cell read)
#   - per-cell donor_id  = decode categories, then index by `codes`
donor_grp = h5["obs"]["donor_id"]

categories = [c.decode() if isinstance(c, bytes) else c
              for c in donor_grp["categories"][:]]
print("unique donor_ids:", categories)

# Per-cell donor_id for all cells (maps each code back to its category):
codes = donor_grp["codes"][:]
donor_per_cell = pd.Categorical.from_codes(codes, categories=categories)
print("n cells:", len(donor_per_cell))
print("counts per donor:")
print(pd.Series(donor_per_cell).value_counts())

unique donor_ids: ['H18.30.001', 'H19.30.004', 'H20.30.001', 'H20.30.002', 'H21.30.004', 'H23.30.001', 'H24.30.001', 'H24.30.003', 'H24.30.004', 'H24.30.007']
n cells: 1034819
counts per donor:
H24.30.004    169312
H24.30.001    151553
H23.30.001    146672
H24.30.003    126324
H24.30.007    115833
H21.30.004    108830
H20.30.001     77759
H20.30.002     74595
H18.30.001     56363
H19.30.004      7578
Name: count, dtype: int64
